# Flow-Matching Speech Enhancement — Colab Training

One-click training notebook for Colab Pro (GPU runtime).

**Prerequisites:**
1. Upload `archives/` folder (from `bash scripts/pack_for_colab.sh`) to your Google Drive root
2. Push the repo to GitHub
3. Select **GPU runtime** (Runtime → Change runtime type → T4/A100)

---

## 0. Setup & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU memory: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB')

## 1. Clone Repo & Install Dependencies

In [ ]:
# ---- CONFIGURE THIS ----
GITHUB_REPO = "your-username/speech-enhancement-project"  # <-- CHANGE THIS
BRANCH = "main"
# ------------------------

import os
PROJECT_DIR = "/content/speech-enhancement-project"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {BRANCH} https://github.com/{GITHUB_REPO}.git {PROJECT_DIR}
else:
    !cd {PROJECT_DIR} && git pull origin {BRANCH}

os.chdir(PROJECT_DIR)
!pip install -q -r requirements.txt

## 2. Unpack Pre-extracted Features

In [ ]:
import os, glob
os.chdir(PROJECT_DIR)

ARCHIVES_DIR = "/content/drive/MyDrive/archives"

# Check if features already unpacked
feat_dir = os.path.join(PROJECT_DIR, "data/features/clean_dac")
if os.path.exists(feat_dir) and len(os.listdir(feat_dir)) > 0:
    print(f"Features already unpacked ({len(os.listdir(feat_dir))} files in clean_dac)")
else:
    assert os.path.exists(ARCHIVES_DIR), (
        f"Archives not found: {ARCHIVES_DIR}\n"
        "Upload the archives/ folder to Google Drive root.\n"
        "(Run: bash scripts/pack_for_colab.sh on your Mac first)"
    )
    # Unpack each per-directory archive
    os.makedirs("data/features", exist_ok=True)
    for archive in sorted(glob.glob(f"{ARCHIVES_DIR}/features_*.tar.gz")):
        print(f"Unpacking {os.path.basename(archive)} ...")
        !tar xzf {archive} -C data/features/
    print("Features unpacked!")

# Verify
for d in ['clean_dac', 'noisy_dac', 'moss_last', 'moss_multi']:
    path = f"data/features/{d}"
    n = len(os.listdir(path)) if os.path.exists(path) else 0
    print(f"  {d}: {n} files")

## 3. Training Configuration

Adjust the config for Colab GPU (larger batch size, more workers).

In [ ]:
import yaml

with open('configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)

# ---- Colab-optimised overrides ----
# Increase batch size for GPU (T4=16GB → bs=32; A100=40GB → bs=64)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1024**3 if torch.cuda.is_available() else 0
if gpu_mem >= 35:
    config['training']['batch_size'] = 64
elif gpu_mem >= 14:
    config['training']['batch_size'] = 32
else:
    config['training']['batch_size'] = 16

config['training']['device'] = 'cuda'

print(f"GPU memory: {gpu_mem:.1f} GB")
print(f"Batch size: {config['training']['batch_size']}")
print(f"Num steps:  {config['training']['num_steps']}")
print(f"Condition:  {config['model']['condition_type']}")

# Save the Colab-tuned config
with open('configs/colab.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("\nSaved configs/colab.yaml")

## 4. Train — Multi-Layer Conditioning (main experiment)

In [ ]:
os.chdir(PROJECT_DIR)
!python train.py --config configs/colab.yaml --condition_type multi_layer

## 5. (Optional) Train ablation variants

Run these if you want the baseline comparisons.

In [ ]:
# Uncomment to run:
# !python train.py --config configs/colab.yaml --condition_type last_layer
# !python train.py --config configs/colab.yaml --condition_type none

## 6. Save Checkpoints to Google Drive

Copy checkpoints back to Drive so they persist after runtime disconnects.

In [ ]:
import shutil

src = os.path.join(PROJECT_DIR, 'checkpoints')
dst = '/content/drive/MyDrive/speech_enhancement_checkpoints'

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Checkpoints saved to {dst}')
    # List saved files
    for root, dirs, files in os.walk(dst):
        for f in files:
            fpath = os.path.join(root, f)
            size_mb = os.path.getsize(fpath) / 1024**2
            print(f'  {os.path.relpath(fpath, dst)}: {size_mb:.1f} MB')
else:
    print('No checkpoints directory found.')

## 7. Evaluate

In [ ]:
# Find the latest checkpoint
import glob

ckpt_pattern = os.path.join(PROJECT_DIR, 'checkpoints/multi_layer/step_*.pt')
ckpts = sorted(glob.glob(ckpt_pattern))

if ckpts:
    latest_ckpt = ckpts[-1]
    print(f'Latest checkpoint: {latest_ckpt}')
    !python evaluate.py --config configs/colab.yaml --checkpoint {latest_ckpt} --condition_type multi_layer
else:
    print('No checkpoints found. Train first.')

## 8. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir checkpoints/